# Batch ODE Kinetic Fitting

Run the full DK → ODE fitting pipeline on all samples in a `.cxw` file.
This notebook demonstrates:
- Loading a `.cxw` experiment file
- Running `batch_fit(mode='ode')` for all samples
- Flagging poor fits and non-specific binders
- Saving individual data-vs-model plots as PNGs
- Exporting a summary CSV

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sensofit import load_cxw, batch_fit, flag_poor_fits
from sensofit.plotting import plot_fit, save_fit_plots

## Configure Paths

Set the input `.cxw` file and output directory for plots and CSV.

In [ ]:
# --- Edit these paths ---
CXW_FILE = '../20260318_EV71 2A Binding assay.cxw'
OUTPUT_DIR = Path('../results/20260318_EV71 2A Binding assay')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input:  {CXW_FILE}')
print(f'Output: {OUTPUT_DIR.resolve()}')

## Run Batch ODE Fitting

Fits every sample cycle using Direct Kinetics → ODE refinement.
Non-specific binders are detected and skipped automatically.

In [ ]:
df, data = batch_fit(CXW_FILE, mode='dk', include_nsb=True, progress=True)
df = flag_poor_fits(df)
print(f'\n{len(df)} samples fitted, {df["flag"].sum()} flagged')

## Results Summary

Display the kinetic parameters table. Flagged rows indicate fits that may need manual review.

In [ ]:
# Key columns for review
cols = ['compound', 'concentration_uM', 'ka', 'kd', 'KD_uM', 'Rmax',
        'sigma_res', 'nonspecific', 'flag', 'flag_reason']
display_cols = [c for c in cols if c in df.columns]
df[display_cols].head(20)

## Save Fit Plots as PNGs

Re-fit each sample individually to get the full result arrays needed for plotting,
then save annotated data-vs-model overlays.

In [ ]:
from sensofit.ode_fitting import fit_sample as ode_fit_sample

samples = data['samples']
results = []

for i, sample in enumerate(samples):
    row = df[df['cycle_index'] == sample['index']]
    if row.empty or row.iloc[0].get('nonspecific', False) or not row.iloc[0].get('success', False):
        results.append(None)
        continue
    try:
        result = ode_fit_sample(sample, data['dmso_cals'], blanks=data['blanks'])
        results.append(result)
    except Exception:
        results.append(None)

plot_dir = str(OUTPUT_DIR / 'plots')
paths = save_fit_plots(results, samples, plot_dir, mode='ode')
n_plots = sum(1 for p in paths if p is not None)
print(f'Saved {n_plots} plot(s) → {plot_dir}/')

## Example Fit Plot

Display one of the generated plots inline.

In [ ]:
# Show first successful fit
%matplotlib inline
for result, sample in zip(results, samples):
    if result is not None:
        fig = plot_fit(result, sample)
        plt.show()
        break

## Export CSV

Save the full results table with source file, cycle number, sample ID and quality flags.

In [ ]:
csv_path = OUTPUT_DIR / 'batch_ode_results.csv'
df.to_csv(csv_path, index=False)
print(f'Results saved → {csv_path}')
print(f'  {len(df)} total samples')
print(f'  {df["nonspecific"].sum()} non-specific binders')
print(f'  {df["flag"].sum()} flagged fits')